In [12]:
import numpy as np

def format_delta_chi2_summary(self):
    """
    Returns a clean, color-formatted, column-aligned string summary of Δχ² statistics
    for accepted and discarded points (NumPy arrays).
    """

    def compute_stats(arr):
        if arr is None or arr.size == 0:
            return None
        return {
            "N": arr.size,
            "Mean": np.mean(arr),
            "Std": np.std(arr),
            "Min": np.min(arr),
            "Max": np.max(arr),
            "Median": np.median(arr),
        }

    stats_accepted = compute_stats(self.delta_chi2_accepted)
    stats_discarded = compute_stats(self.delta_chi2_discarded)

    # Define column order and spacing
    columns = ["N", "Mean", "Std", "Min", "Max", "Median"]
    col_width = 12
    label_width = 14

    # Header
    header = " " * label_width + "".join(f"{col:>{col_width}}" for col in columns)

    def format_float_for_table(val, width=10):
        """
        Formats the float `val` so that it fits nicely
        in `width` columns, using either
        - scientific notation when |val| >= 1e5, or
        - normal float with one decimal place otherwise.
        """
        if abs(val) >= 1e5:
            # scientific notation with 1 decimal
            return f"{val:{width}.3e}"
        else:
            # normal float with 1 decimal
            return f"{val:{width}.2f}"

    def format_row(label, stats, threshold=None, accepted=False, discarded=False):
        if stats is None:
            return f"{label:<{label_width}}" + "  [no data]".rjust(col_width)

        # Choose base color
        if discarded:
            base_color = "\033[1;31m"  # red
        elif accepted:
            base_color = "\033[1;32m"  # green (used for label and N)
        else:
            base_color = ""

        reset = "\033[0m"
        row = f"{base_color}{label:<{label_width}}{stats['N']:{col_width}d}{reset}"

        # Other columns
        for col in columns[1:]:  # skip 'N' (already printed)
            val = stats[col]

            # Accepted: check if stat exceeds threshold
            if accepted:
                if threshold is not None and val > threshold:
                    color = "\033[1;33m"  # yellow
                else:
                    color = "\033[1;32m"  # green
            elif discarded:
                color = "\033[1;31m"  # red
            else:
                color = ""

            formatted_val = format_float_for_table(val, width=col_width)
            row += f"{color}{formatted_val}{reset}"

        return row

    # Compose table
    title = "\033[1;4;37mΔχ² Summary Statistics:\033[0m"

    summary = title + "\n"
    summary += header + "\n"
    summary += (
        format_row(
            "Accepted",
            stats_accepted,
            threshold=self.delta_chi2_threshold,
            accepted=True,
        )
        + "\n"
    )
    summary += format_row("Discarded", stats_discarded, discarded=True) + "\n"

    return summary


In [13]:
#Test the above function by creating two NumPy arrays; one for accepted points and one for discarded points.

import numpy as np

# Create some dummy data
np.random.seed(42)
delta_chi2_accepted = np.random.chisquare(3, 100)
#delta_chi2_discarded = np.random.chisquare(3, 50)

#Compute large chi2 values for discarded points
delta_chi2_discarded = np.random.chisquare(3, 50) + 100000


# Create a dummy object to hold the data
class DummyObject:
    pass

dummy = DummyObject()
dummy.delta_chi2_accepted = delta_chi2_accepted
dummy.delta_chi2_discarded = delta_chi2_discarded
dummy.delta_chi2_threshold = 10

# Call the function
print(format_delta_chi2_summary(dummy))

#Output
'''
Δχ² Summary Statistics:
            N      Mean       Std       Min       Max    Median
Accepted      100.00      3.00      2.51      0.07     2.31
Discarded      50.00      3.00      2.51      0.07     2.31
'''



Δχ² Summary Statistics:
                         N        Mean         Std         Min         Max      Median
Accepted               100        2.88        2.12        0.16       12.72        2.48
Discarded               50   1.000e+05        2.33   1.000e+05   1.000e+05   1.000e+05



'\nΔχ² Summary Statistics:\n            N      Mean       Std       Min       Max    Median\nAccepted      100.00      3.00      2.51      0.07     2.31\nDiscarded      50.00      3.00      2.51      0.07     2.31\n'

In [17]:
               print(
                    """
                    \033[1;33mNOTICE:\033[0m The likelihood-filter is applied twice within the same iteration because of strict filtering after reaching the 'bad state' limit. 
                    It is appending new discarded P(k) data to an existing discard file. 
                    This will produce multiple '# z =' blocks for the same redshift. 
                    If you need to parse these discards later, be aware of the duplicated blocks. You can easily fix this manually by moving the blocks to the correct position.
                    This will not affect CONNECT's iterative process or the 'accepted' files.
                    So you can ignore this message if you are not using the discarded P(k) files for further analysis.
                    """
                )


     NOTICE: The likelihood-filter is applied twice within the same iteration because of strict filtering after reaching the 'bad state' limit. 
     It is appending new discarded P(k) data to an existing discard file. 
     This will produce multiple '# z =' blocks for the same redshift. 
     If you need to parse these discards later, be aware of the duplicated blocks. You can easily fix this manually by moving the blocks to the correct position.
     This will not affect CONNECT's iterative process or the 'accepted' files.
     So you can ignore this message if you are not using the discarded P(k) files for further analysis.
     


In [29]:
import numpy as np

# Set up dummy data
np.random.seed(42)
delta_chi2_accepted = np.random.chisquare(3, 100)
delta_chi2_discarded = np.random.chisquare(3, 50)

print(f"Applying likelihood filter to data")
# Define dummy object with needed attributes
class Dummy:
    pass


dummy = Dummy()
dummy.delta_chi2_accepted = delta_chi2_accepted
dummy.delta_chi2_discarded = delta_chi2_discarded
dummy.delta_chi2_threshold = 10
dummy.min_points_reached = False
dummy.N_samples = 150
dummy.min_points_to_keep = 100
dummy.iter_num = 2
dummy.param = type("Param", (), {"discard_worst_first": False})()
dummy.verbosity_level = 2
dummy.format_delta_chi2_summary = lambda: format_delta_chi2_summary(dummy)

# Simulated discarded indices and values
discard_indices = np.random.choice(150, 50, replace=False)
num_filtered = len(discard_indices)
total_potential_discard = 57  # example total that exceeded threshold

# Derived print values
iteration = f"iteration {dummy.iter_num}" if dummy.iter_num > 0 else "initial sampling"
formatted_best_fit = f"{2.3456:.1f}"  # dummy value
filter_method = (
    "Discarded worst points first."
    if dummy.param.discard_worst_first
    else "Filtered sequentially, discarding older iteration points first."
)
if dummy.delta_chi2_accepted is not None and dummy.delta_chi2_accepted.size > 0:
    worst_delta_chi2_accepted = np.max(dummy.delta_chi2_accepted)
else:
    worst_delta_chi2_accepted = 0

delta_chi2_summary = dummy.format_delta_chi2_summary()
delta_chi2_string = (
    f"Δχ²-threshold: {dummy.delta_chi2_threshold}."
    if dummy.delta_chi2_threshold < 99999
    else f"Δχ²-threshold: {dummy.delta_chi2_threshold:.1e}."
)

# Indented multi-line print
indent = "    "  # 4-space indent
summary_lines = [
    delta_chi2_string,
    f"Likelihood-filter discarded {num_filtered}/{dummy.N_samples} points in {iteration} ({total_potential_discard} exceeded threshold).",
    (
        f"Stopped early to retain ≥{dummy.min_points_to_keep} points."
        if dummy.min_points_reached
        else ""
    ),
    (
        f"Worst 'accepted' Δχ²: {worst_delta_chi2_accepted:.1f}."
        if dummy.min_points_reached
        else ""
    ),
    filter_method,
    f"Current global best-fit -log(lkl): {formatted_best_fit}.",
]

# Print
if dummy.verbosity_level > 0:
    for line in summary_lines:
        if line:
            print(f"{indent}{line}")

if dummy.verbosity_level > 1:
    print(f"")
    for line in delta_chi2_summary.split("\n"):
        print(f"{indent}{line}")

Applying likelihood filter to data
    Δχ²-threshold: 10.
    Likelihood-filter discarded 50/150 points in iteration 2 (57 exceeded threshold).
    Filtered sequentially, discarding older iteration points first.
    Current global best-fit -log(lkl): 2.3.

    Δχ² Summary Statistics:
                         N      Mean       Std       Min       Max    Median
    Accepted           100      2.88      2.12      0.16     12.72      2.48
    Discarded           50      2.96      2.33      0.18     11.76      2.45
    


In [2]:
"""
simulate this creation of the headers:

    param_header = "# "
    for par_name in param_names:
        if par_name == param_names[-1]:
            param_header += par_name + "\n"
        else:
            param_header += par_name + "\t"

    derived_header = "# "
    for der_name in param.output_derived:
        if der_name == param.output_derived[-1]:
            derived_header += der_name + "\n"
        else:
            derived_header += der_name + "\t"


    loglkl_header = "# true_loglkl\tchain_loglkl\n"

"""


param_names = ["param1", "param2", "param3"]
param_header = "# "
for par_name in param_names:
    if par_name == param_names[-1]:
        param_header += par_name + "\n"
    else:
        param_header += par_name + "\t"
        
derived_names = ["derived1", "derived2", "derived3"]
derived_header = "# "
for der_name in derived_names:
    if der_name == derived_names[-1]:
        derived_header += der_name + "\n"
    else:
        derived_header += der_name + "\t"
        
loglkl_header = "# true_loglkl\tchain_loglkl\n"


oversampling_header = (
    "# "
    + "\t".join(
        filter(
            None,
            [
                loglkl_header.strip()[1:].strip(),
                param_header.strip()[1:].strip(),
                derived_header.strip()[1:].strip(),
            ],
        )
    )
    + "\n"
)

print(oversampling_header)

# true_loglkl	chain_loglkl	param1	param2	param3	derived1	derived2	derived3



In [13]:
delta_chi2_threshold = 30000



label = (
    r"$\Delta \chi^2{\rm -threshold} = %s$" % delta_chi2_threshold
)

print(label)

$\Delta \chi^2{\rm -threshold} = 30000$


In [65]:
import os
import re


def restore_logs_selective(
    project_path, output_filename="output.log", restored_filename="output_restored.log"
):
    """
    Scans each training.log in ascending iteration order, extracting *all* lines between
    a line beginning with "Test loss:" and the next boundary.

    A boundary is defined as:
      - A line that contains "[=" anywhere in its text and does NOT contain either
        "Training neural network" or "Final model". (This progress line is not captured.)
      - A line that contains "Training neural network" (this line is included),
      - A line that contains "Final model" (this line is included along with the immediately following line,
        assumed to be a timestamp).

    The collected blocks are appended to a newly created file (restored_filename)
    after the existing lines in output_filename.

    :param project_path: Path to the main project folder.
    :param output_filename: The name of the incomplete main log (default 'output.log').
    :param restored_filename: Name of the new combined file (default 'output_restored.log').
    """

    main_log_path = os.path.join(project_path, output_filename)
    restored_path = os.path.join(project_path, restored_filename)

    # 1) Read the original partial output.log lines
    if not os.path.isfile(main_log_path):
        print(f"[restore_logs_selective] No file '{main_log_path}' found. Aborting.")
        return
    with open(main_log_path, "r", encoding="utf-8", errors="replace") as f:
        main_lines = f.readlines()

    # 2) Create (or overwrite) restored file by writing the original lines
    
    #delete the restored file if it already exists
    if os.path.exists(restored_path):
        os.remove(restored_path)
    
    with open(restored_path, "w", encoding="utf-8") as f_out:
        f_out.writelines(main_lines)
        f_out.write("\n[Log restoration: appended lines from training.log below]\n\n")

    # 3) Identify iteration folders in ascending order
    def parse_iter_index(name):
        """Return numeric iteration index or None if not recognized."""
        if name.startswith("number_"):
            try:
                return int(name.split("_")[-1])
            except:
                return None
        if name.startswith("N-"):
            return 0
        return None

    all_subdirs = sorted(os.listdir(project_path))
    iteration_dirs = []
    for d in all_subdirs:
        full_path = os.path.join(project_path, d)
        if not os.path.isdir(full_path):
            continue
        idx = parse_iter_index(d)
        if idx is not None:
            iteration_dirs.append((idx, d))
    iteration_dirs.sort(key=lambda x: x[0])

    # 4) For each iteration folder, parse training.log and append captured blocks.
    for iteration_idx, folder_name in iteration_dirs:
        tlog_path = os.path.join(project_path, folder_name, "training.log")
        if not os.path.isfile(tlog_path):
            continue

        print(f"[restore_logs_selective] Processing {folder_name}/training.log ...")

        blocks = extract_blocks(tlog_path)
        if not blocks:
            print("    -> no relevant blocks found.")
            continue

        with open(restored_path, "a", encoding="utf-8") as f_out:
            f_out.write(f"\n\n[Appending logs from {folder_name}/training.log]\n")
            for block in blocks:
                for line in block:
                    f_out.write(line)

        total_lines = sum(len(b) for b in blocks)
        print(f"    -> appended {total_lines} lines from {tlog_path}.")

    print(f"\nDone. Combined log saved to:\n    {restored_path}")


def extract_blocks(training_log_path):
    """
    Extracts text blocks from a training.log file.

    A block is defined as all lines starting from a line that begins with "Test loss:"
    up to the next boundary. A boundary is:
      - A line that contains "[=" anywhere in its text and does NOT contain either
        "Training neural network" or "Final model". (This progress line is NOT captured.)
      - A line that contains "Training neural network" (this line IS captured), or
      - A line that contains "Final model" (this line IS captured and the immediately following
        line is also captured, which is assumed to be the timestamp).

    Returns a list of blocks, each block is a list of lines (with newlines intact).
    """
    blocks = []
    in_capture = False
    current_block = []

    test_loss_start = re.compile(r"^Test loss:\s*")
    training_net_str = "Training neural network"
    final_model_str = "Final model"

    with open(training_log_path, "r", encoding="utf-8", errors="replace") as f_in:
        for line in f_in:
            if not in_capture:
                if test_loss_start.search(line):
                    in_capture = True
                    current_block = [line]
            else:
                # Check if this line is a progress boundary:
                if (
                    "[=" in line
                    and (training_net_str not in line)
                    and (final_model_str not in line)
                ):
                    # End the block without including this progress line.
                    blocks.append(current_block)
                    in_capture = False
                    current_block = []
                elif final_model_str in line:
                    current_block.append(line)
                    # Capture the very next line (presumed to be a timestamp)
                    next_line = f_in.readline()
                    if next_line:
                        current_block.append(next_line)
                    blocks.append(current_block)
                    in_capture = False
                    current_block = []
                elif training_net_str in line:
                    current_block.append(line)
                    blocks.append(current_block)
                    in_capture = False
                    current_block = []
                else:
                    current_block.append(line)

        if in_capture and current_block:
            blocks.append(current_block)

    return blocks


# Example usage:
project_path = (
    "/home/maanson/Speciale/connectv2/data/dcdm/dcdm_baseline_filter_thres500"
)
restore_logs_selective(project_path)

[restore_logs_selective] Processing N-10000/training.log ...
    -> no relevant blocks found.
[restore_logs_selective] Processing number_1/training.log ...
    -> appended 32 lines from /home/maanson/Speciale/connectv2/data/dcdm/dcdm_baseline_filter_thres500/number_1/training.log.
[restore_logs_selective] Processing number_2/training.log ...
    -> appended 32 lines from /home/maanson/Speciale/connectv2/data/dcdm/dcdm_baseline_filter_thres500/number_2/training.log.
[restore_logs_selective] Processing number_3/training.log ...
    -> appended 55 lines from /home/maanson/Speciale/connectv2/data/dcdm/dcdm_baseline_filter_thres500/number_3/training.log.
[restore_logs_selective] Processing number_4/training.log ...
    -> appended 40 lines from /home/maanson/Speciale/connectv2/data/dcdm/dcdm_baseline_filter_thres500/number_4/training.log.
[restore_logs_selective] Processing number_5/training.log ...
    -> appended 40 lines from /home/maanson/Speciale/connectv2/data/dcdm/dcdm_baseline_filte

In [58]:
import os
import sys
import pandas as pd
import numpy as np

# =============================================================================
# 1. Load the Parameters from the actual .param file using CONNECT's Parameters class
# =============================================================================
# Set the paths (adjust if needed)
CONNECT_PATH = "/home/maanson/Speciale/connectv2"
DATA_PATH = "data/dcdm/dcdm_baseline_filter_thres500"
PARAM_FILE = os.path.join(CONNECT_PATH, DATA_PATH, "log_connect.param")

# Import the Parameters class from CONNECT's source code.
# (Make sure that the module "source.default_module" is on your PYTHONPATH.)
from source.default_module import Parameters

param = Parameters(PARAM_FILE)

# =============================================================================
# 2. Create a dummy CONNECT object that uses the actual load_data_file and check_likelihood_filter_health functions
# =============================================================================
# (These functions are assumed to be exactly as in your CONNECT source code.)
# You can copy them verbatim from your code if they are not already imported.
# For this example, we assume they are defined below.


# --- Provided function: compare_dataframes ---

def load_data_file(self, file_path, verbose=1):
    """
    Load a data file that has a header line starting with '#' and returns a DataFrame.
    """
    if not os.path.isfile(file_path):
        if verbose >= 1:
            print(f"[load_data_file] File {file_path} does not exist.", flush=True)
        return None

    header_line = None
    with open(file_path, "r") as f:
        for line in f:
            if line.startswith("#"):
                header_line = line.lstrip("#").strip()
                break

    if header_line is None:
        raise ValueError(f"No header line starting with '#' found in {file_path}")

    columns = header_line.split()
    if verbose >= 3:
        print(f"[load_data_file] Columns for {file_path}: {columns}", flush=True)

    df = pd.read_csv(
        file_path,
        sep=r"\s+",
        comment="#",
        names=columns,
        index_col=False,
        dtype=np.float32,
    )

    # Optional sanity checks
    if df.empty and verbose >= 2:
        print(
            f"[load_data_file] Warning: Loaded DataFrame from {file_path} is empty.",
            flush=True,
        )

    return df


def compare_dataframes(
    df1,
    df2,
    df_likelihood=None,
    df_likelihood2=None,
    comparison_type="new",
    verbose=1,
    compare_context=None,
):

    import pandas as pd

    """
    
    #######
    This function was initially developed for the 'plot_iterations.py' module used to analyze the iterative sampling process.
    But it can be used here as well to compare the data overlap between the final accepted data by the likelihood-filter and the percentage of new data from the chains.
    This gives information of how large percentage of the data is actually accepted, and helps us track if the likelihood-filter is too strict for the procedure to converge naturally.
    #######

    Compare two DataFrames (df1, df2) to identify samples that are 'new', 'removed', or 'common',
    while preserving one-to-one matching of duplicates. Also re-aligns likelihood data if provided.

    Parameters
    ----------
    df1 : pd.DataFrame
        The first DataFrame (e.g., the "current" iteration's data).
    df2 : pd.DataFrame
        The second DataFrame (e.g., the "previous" iteration's data).
    df_likelihood : pd.DataFrame, optional
        Likelihood rows aligned with df1 (same length, same row order as df1 BEFORE sorting).
    df_likelihood2 : pd.DataFrame, optional
        Likelihood rows aligned with df2 (same length, same row order as df2 BEFORE sorting).
    comparison_type : {'new','removed','common'}, default='new'
        - 'new': return rows in df1 that are not in df2.
        - 'removed': return rows in df2 that are not in df1.
        - 'common': return rows present in both df1 and df2.
    verbose : int, optional
        If >0, prints some debugging info.

    Returns
    -------
    matched_params : pd.DataFrame
        Subset of parameter rows that match the requested relationship,
        extracted from the correct perspective (df1 or df2, or intersection).
    matched_likelihood : pd.DataFrame or None
        Subset of likelihood rows that align with matched_params. If none given,
        returns None.
    """

    # --------------------------------------------------
    # Step 1: Validate input parameters
    # --------------------------------------------------
    valid_types = ["new", "removed", "common"]
    if comparison_type not in valid_types:
        raise ValueError(
            f"comparison_type must be one of {valid_types}, got: {comparison_type}"
        )

    # --------------------------------------------------
    # Step 2: Handle edge cases (if df1 or df2 is empty)
    # --------------------------------------------------
    if df1 is None or df1.empty:
        if verbose > 0:
            print(
                f'\n[compare_dataframes] [{compare_context["context"]}] df1 ({compare_context["df1"]}) is empty; returning trivial result:\n {compare_context["msg1"]}'
            )
        if comparison_type == "removed" and df2 is not None:
            return df2.copy().reset_index(drop=True), df_likelihood2
        return None, None

    if df2 is None or df2.empty:
        if verbose > 0:
            print(
                f'\n[compare_dataframes] [{compare_context["context"]}] df2 ({compare_context["df2"]}) is empty; returning trivial result:\n {compare_context["msg2"]}'
            )
        if comparison_type == "new":
            return df1.copy().reset_index(drop=True), df_likelihood
        return None, None

    # --------------------------------------------------
    # Step 2b: Ensure likelihood data and dataframes match in length
    if df_likelihood is not None and len(df_likelihood) != len(df1):
        raise ValueError(
            f'\nLength mismatch: df_likelihood ({len(df_likelihood)}) and df1 ({compare_context["df1"]}) ({len(df1)}) are not equal.'
        )
    if df_likelihood2 is not None and len(df_likelihood2) != len(df2):
        raise ValueError(
            f'\nLength mismatch: df_likelihood2 ({len(df_likelihood2)}) and df2 ({compare_context["df2"]}) ({len(df2)}) are not equal.'
        )

    # --------------------------------------------------
    # Step 3: Preprocess df1 (current iteration's data)
    # --------------------------------------------------
    df1 = df1.copy()
    df1["_temp_idx1"] = df1.index  # Store original row index before sorting

    # Identify the relevant parameter columns (excluding helper columns)
    param_cols = [c for c in df1.columns if c not in ["_temp_idx1", "dup_id"]]

    # Sort df1 so that identical samples appear together
    df1_sorted = df1.sort_values(param_cols, kind="mergesort").reset_index(drop=True)

    # Assign a 'dup_id' to each duplicate row so they can be matched one-to-one
    df1_sorted["dup_id"] = df1_sorted.groupby(param_cols).cumcount()

    # Reorder the likelihood data to match this new sorted order
    df_likelihood_sorted = (
        df_likelihood.iloc[df1_sorted["_temp_idx1"]].reset_index(drop=True)
        if df_likelihood is not None
        else None
    )

    # --------------------------------------------------
    # Step 4: Preprocess df2 (previous iteration's data)
    # --------------------------------------------------
    df2 = df2.copy()
    df2["_temp_idx2"] = df2.index

    df2_sorted = df2.sort_values(param_cols, kind="mergesort").reset_index(drop=True)
    df2_sorted["dup_id"] = df2_sorted.groupby(param_cols).cumcount()

    df_likelihood2_sorted = (
        df_likelihood2.iloc[df2_sorted["_temp_idx2"]].reset_index(drop=True)
        if df_likelihood2 is not None
        else None
    )

    # --------------------------------------------------
    # Step 5: Perform Merge to Find Matches
    # --------------------------------------------------
    """
    We now compare df1_sorted and df2_sorted to determine which samples belong to which category:
    
    - 'new': Samples in df1 but not in df2 (found using a LEFT JOIN)
    - 'removed': Samples in df2 but not in df1 (found using a RIGHT JOIN)
    - 'common': Samples that exist in both df1 and df2 (found using an INNER JOIN)

    The 'merge' function combines both dataframes based on their common parameter columns + 'dup_id'.
    This ensures that duplicate rows match correctly and one-to-one.
    
    The 'how' parameter controls which type of comparison we perform:
    
    - 'left' (for 'new'): Keeps all rows from df1_sorted, adds matches from df2_sorted.
    - 'right' (for 'removed'): Keeps all rows from df2_sorted, adds matches from df1_sorted.
    - 'inner' (for 'common'): Keeps only rows that exist in BOTH df1_sorted and df2_sorted.

    The 'indicator=True' adds a new column `_merge`, which labels each row as:
    - 'left_only'  → Present only in df1 (new sample)
    - 'right_only' → Present only in df2 (removed sample)
    - 'both'       → Present in both (common sample)
    """
    if comparison_type == "new":
        merge_type = "left"
        indicator = True
    elif comparison_type == "removed":
        merge_type = "right"
        indicator = True
    else:  # 'common'
        merge_type = "inner"
        indicator = False

    merged = df1_sorted.merge(
        df2_sorted,
        on=param_cols + ["dup_id"],
        how=merge_type,
        indicator=indicator,
        suffixes=("_df1", "_df2"),
    )

    # --------------------------------------------------
    # Step 6: Extract the Matching Rows from the Merge
    # --------------------------------------------------
    """
    Now that we have merged df1_sorted and df2_sorted, we extract the rows based on `_merge`:

    - For 'new': We filter only rows labeled as 'left_only' (i.e., samples that appear in df1 but not df2).
    - For 'removed': We filter only rows labeled as 'right_only' (samples in df2 but not df1).
    - For 'common': We take all merged rows, since they exist in both dataframes.
    """
    if comparison_type == "new":
        matched_df = merged[merged["_merge"] == "left_only"].drop(columns=["_merge"])
    elif comparison_type == "removed":
        matched_df = merged[merged["_merge"] == "right_only"].drop(columns=["_merge"])
    else:  # 'common'
        matched_df = merged

    # Extract the original rows from df1 or df2
    matched_params = (
        df1.iloc[matched_df["_temp_idx1"]].copy()
        if comparison_type != "removed"
        else df2.iloc[matched_df["_temp_idx2"]].copy()
    )

    # Remove helper columns
    matched_params.drop(
        columns=["dup_id", "_temp_idx1", "_temp_idx2"],
        inplace=True,
        errors="ignore",
    )
    matched_params.reset_index(drop=True, inplace=True)

    # --------------------------------------------------
    # Step 7: Extract Aligned Likelihood Data
    # --------------------------------------------------
    matched_likelihood = None
    if comparison_type in ["new", "common"] and df_likelihood_sorted is not None:
        matched_likelihood = (
            df_likelihood.iloc[matched_df["_temp_idx1"]].copy().reset_index(drop=True)
        )
    elif comparison_type == "removed" and df_likelihood2_sorted is not None:
        matched_likelihood = (
            df_likelihood2.iloc[matched_df["_temp_idx2"]].copy().reset_index(drop=True)
        )

    return matched_params, matched_likelihood

    # --- Provided function: check_likelihood_filter_health ---

def check_likelihood_filter_health(
    self,
    N_accepted,
    N_in_data_set,
    i,
    mcmc,
):

    # ---------------- 2nd CONVERGENCE CHECK ----------------
    # Import model_params from current iteration. The accepted points after the likelihood filter
    all_accepted_after_lklfilter_df = self.load_data_file(
        os.path.join(
            self.CONNECT_PATH,
            self.data_path,
            f"number_{i}",
            "model_params.txt",
        ),
        verbose=2,
    )
    all_accepted_after_lklfilter_likelihood_df = self.load_data_file(
        os.path.join(
            self.CONNECT_PATH,
            self.data_path,
            f"number_{i}",
            "likelihood_data.txt",
        ),
        verbose=2,
    )

    # Import data from chains; i.e. data accepted from oversampling filter
    accepted_by_oversampling = mcmc.import_points_from_chains(i)

    param_cols = all_accepted_after_lklfilter_df.columns
    accepted_by_oversampling_df = pd.DataFrame(
        accepted_by_oversampling, columns=param_cols
    )

    # Compare the two dataframes to find the overlap
    compare_context = {
        "context": "Finding overlap between points accepted by likelihood filter and new points from chains",
        "df1": f"all_accepted_after_lklfilter",
        "df2": f"accepted_by_oversampling",
        "msg1": "all_accepted_after_lklfilter is empty",
        "msg2": "accepted_by_oversampling is empty",
    }
    # New data that survived the oversampling filter and also survived the likelihood filter. i.e. all new points added to the accepted pool this iteration.
    new_accepted_df, new_accepted_likelihood_df = compare_dataframes(
        df1=all_accepted_after_lklfilter_df,
        df2=accepted_by_oversampling_df,
        df_likelihood=all_accepted_after_lklfilter_likelihood_df,
        comparison_type="common",  # Find common points between the two dataframes
        compare_context=compare_context,
        verbose=1,
    )

    N_accepted_after_lkl_filter = len(new_accepted_df)
    print(
        f"    The likelihood-filter accepted {N_accepted_after_lkl_filter}/{N_accepted} new points of the points that survived the oversampling filter.",
        flush=True,
    )

    if (
        N_accepted > 0.1 * N_in_data_set
        and N_accepted_after_lkl_filter < 0.1 * N_in_data_set
    ):
        self.consecutive_bad_states_count += 1

        if self.consecutive_bad_states_count >= 3:

            print(
                f"""
                ────────────────────────────────────────────────────────────────────
                ⚠️  \033[1;31mWARNING:\033[0m Iterative process may have entered a "bad state".
                ────────────────────────────────────────────────────────────────────
                
                🔁 The "bad state" has occured for the last {self.consecutive_bad_states_count} consecutive iterations.
                
                The likelihood filter might be too agressive:
                - Only {N_accepted_after_lkl_filter/N_in_data_set*100:.2f}% of the new data from the chains was accepted *after* applying both filters (oversampling- and likelihood-filter).
                - Yet, {N_accepted/N_in_data_set*100:.2f}% > 10% of the newly generated data was accepted by the oversampling filter.
                
                ⚠️  Risk:
                This indicates that the likelihood filter MIGHT be too agressive and the MCMC sampling might continue to generate new data that is rejected by the likelihood filter.
                This can lead to a situation where the MCMC sampling is stuck in a infinite loop, where it keeps generating the same data being discarded by the likelihood filter.
                In a nutshell, the MCMC sampler might keep sampling points in the same region and too many points are outside the likelihood filter's acceptance region.
                Hence, CONNECT might never converge naturally by reaching the convergence criteria: 'Oversampling filter accepting <10% of new data'.
                
                ✅ Recommendation:
                If this keeps happening for multiple consecutive iterations, you might want to consider increasing the threshold for the likelihood-filter or disable it completely. 
                Analyze the iterative outputs carefully: You may use the associated the "plot_iterations.py" tool.
                You might want to try and run CONNECT with a Δχ²-threshold = 1e+32, keep_intial_data=False, keep_first_iteration=False, and use_likelihood_filter=True, to emulate a normal CONNECT run without the likelihood filter.
                Then you can analyze the Δχ²-distribution and consider how agressive you need the likelihood filter to be.
                
                🔍This "bad state" is not a definitive indicator that the trained neural network is a bad emulator diverging from the true posterior distribution.
                But you need to proceed with caution and analyze the situation carefully if it keeps occuring. An overly agressive filter have previously caused poor results.
                
                ⏳ CONNECT will continue to run the iterative sampling for minimum {self.param.max_consecutive_bad_states - self.consecutive_bad_states_count} iterations more, to see if the situation improves.
                
                """,
                flush=True,
            )
            print(
                f"""
                ────────────────────────────────────────────────────────────────────
                ⚠️  WARNING: Iterative process may have entered a "bad state".
                ────────────────────────────────────────────────────────────────────
                The likelihood filter might be too agressive. 
                Please check the output.log in CONNECT's data folder under this running project for more information.
                """,
                flush=True,
                file=sys.stderr,
            )

            if self.param.auto_update_threshold:

                print(
                    f"    CONNECT is run with auto_update_threshold=True. Updating the likelihood-filter threshold to avoid the 'bad state'.",
                    flush=True,
                )
                print(
                    f"    The likelihood filter threshold is currently set to: {self.param.delta_chi2_threshold}",
                    flush=True,
                )
                print(
                    f"    The likelihood filter threshold will be set to 12.5th percentile of the delta_chi2 values of the accepted points by the oversampling-filter.",
                    flush=True,
                )

                # Update the threshold for the filter to be less agressive, above the lowest 10% delta_chi2 value of the accepted points by the oversampling-filter.
                # Load discarded data from the likelihood filter
                discarded_by_lklfilter_df = self.load_data_file(
                    os.path.join(
                        self.CONNECT_PATH,
                        self.data_path,
                        f"number_{i}",
                        "loglkl_discarded_data",
                        "model_params.txt",
                    ),
                    verbose=2,
                )
                discarded_by_lklfilter_likelihood_df = self.load_data_file(
                    os.path.join(
                        self.CONNECT_PATH,
                        self.data_path,
                        f"number_{i}",
                        "loglkl_discarded_data",
                        "likelihood_data.txt",
                    ),
                    verbose=2,
                )

                # Compare the discarded data from the likelihood filter with the new data from the chains
                compare_context = {
                    "context": "Finding overlap between points discarded by likelihood filter and new points from chains",
                    "df1": f"discarded_by_lklfilter",
                    "df2": f"accepted_by_oversampling",
                    "msg1": "discarded_by_lklfilter is empty",
                    "msg2": "accepted_by_oversampling is empty",
                }

                # New data that survived the oversampling filter but was discarded by the likelihood filter. i.e. all points discarded by the likelihood filter this iteration, which was generated by the chains.
                new_discarded_df, new_discarded_likelihood_df = compare_dataframes(
                    df1=discarded_by_lklfilter_df,
                    df2=accepted_by_oversampling_df,
                    df_likelihood=discarded_by_lklfilter_likelihood_df,
                    comparison_type="common",
                    compare_context=compare_context,
                    verbose=1,
                )

                # Find lowest loglkl value in all_accepted_after_lklfilter_likelihood_df
                bestfit_loglkl = all_accepted_after_lklfilter_likelihood_df[
                    "true_loglkl"
                ].min()

                # Now append the new_discarded_likelihood_df and new_accepted_likelihood_df into one dataframe: accepted_by_oversampling_likelihood_df

                accepted_by_oversampling_likelihood_df = pd.concat(
                    [new_discarded_likelihood_df, new_accepted_likelihood_df]
                )

                # Now compute the delta_chi2 values for accepted_by_oversampling_likelihood_df["true_loglkl"] and bestfit_loglkl
                delta_chi2 = 2 * (
                    accepted_by_oversampling_likelihood_df["true_loglkl"]
                    - bestfit_loglkl
                )

                # Compute the 12.5th percentile of the delta_chi2 values
                threshold = np.percentile(delta_chi2, 12)

                # Update the likelihood filter threshold
                self.param.delta_chi2_threshold = threshold
                print(
                    f"    The likelihood filter threshold has been updated to: {self.param.delta_chi2_threshold}",
                    flush=True,
                )

    else:
        self.consecutive_bad_states_count = 0


# Now, define a dummy CONNECT-like object that uses the real functions above.
class DummyCONNECT:
    def __init__(self, CONNECT_PATH, data_path, param):
        self.CONNECT_PATH = CONNECT_PATH
        self.data_path = data_path
        self.param = param
        self.param.auto_update_threshold = True
        # Set this to trigger the threshold update branch.
        self.consecutive_bad_states_count = 4

    # Attach the functions as methods
    load_data_file = load_data_file
    check_likelihood_filter_health = check_likelihood_filter_health


# =============================================================================
# 3. Instantiate the dummy CONNECT object and the real mcmc sampler
# =============================================================================
dummy_connect = DummyCONNECT(CONNECT_PATH, DATA_PATH, param)

# Instantiate the mcmc sampler using your exec method (this uses the actual mcmc sampler)
exec(f"from source.mcmc_samplers.{param.mcmc_sampler} import {param.mcmc_sampler}")
_locals = {}
exec(
    f"mcmc = {param.mcmc_sampler}(param, CONNECT_PATH)",
    globals(),
    _locals,
)
mcmc = _locals["mcmc"]

# =============================================================================
# 4. Run check_likelihood_filter_health with chosen parameters
# =============================================================================

#N_accepted = mcmc.discard_oversampled_points(8)
chain_data = mcmc.import_points_from_chains(8)
N_accepted = len(chain_data)
N_in_data_set = mcmc.get_number_of_data_points(8 - 1) + N_accepted


# For example, set N_accepted=15 and N_in_data_set=100 to force the branch where the threshold is updated.
dummy_connect.check_likelihood_filter_health(
    N_accepted=N_accepted, N_in_data_set=N_in_data_set, i=8, mcmc=mcmc
)

# Finally, print the updated threshold:
print("\nFinal likelihood filter threshold:", dummy_connect.param.delta_chi2_threshold)

    The likelihood-filter accepted 1443/4177 new points of the points that survived the oversampling filter.

                ────────────────────────────────────────────────────────────────────
                ⚠️  WARNING: Iterative process may have entered a "bad state".
                ────────────────────────────────────────────────────────────────────
                
                🔁 The "bad state" has occured for the last 5 consecutive iterations.
                
                The likelihood filter might be too agressive:
                - Only 4.95% of the new data from the chains was accepted *after* applying both filters (oversampling- and likelihood-filter).
                - Yet, 14.32% > 10% of the newly generated data was accepted by the oversampling filter.
                
                ⚠️  Risk:
                This indicates that the likelihood filter MIGHT be too agressive and the MCMC sampling might continue to generate new data that is rejected by the likeli


                ────────────────────────────────────────────────────────────────────
                ⚠️  WARNING: Iterative process may have entered a "bad state".
                ────────────────────────────────────────────────────────────────────
                The likelihood filter might be too agressive. 
                Please check the output.log in CONNECT's data folder under this running project for more information.
                


    CONNECT is run with auto_update_threshold=True. Updating the likelihood-filter threshold to avoid the 'bad state'.
    The likelihood filter threshold is currently set to: 500
    The likelihood filter threshold will be set to 12.5th percentile of the delta_chi2 values of the accepted points by the oversampling-filter.
    The likelihood filter threshold has been updated to: 968.5527050781251

Final likelihood filter threshold: 968.5527050781251


In [34]:
import os
import sys
import pandas as pd
import numpy as np

    # =============================================================================
    # 1. Load the Parameters from the actual .param file using CONNECT's Parameters class
    # =============================================================================
    # Set the paths (adjust if needed)
    
def write_acceptance_rate_to_log(project_name, output_filename="output_restored.log"):
    
    
    
    CONNECT_PATH = "/home/maanson/Speciale/connectv2"
    #DATA_PATH = "data/dcdm/dcdm_baseline_filter_thres500"
    DATA_PATH = f"data/dcdm/{project_name}"
    PARAM_FILE = os.path.join(CONNECT_PATH, DATA_PATH, "log_connect.param")

    # Import the Parameters class from CONNECT's source code.
    # (Make sure that the module "source.default_module" is on your PYTHONPATH.)
    from source.default_module import Parameters

    param = Parameters(PARAM_FILE)

    # =============================================================================
    # 2. Create a dummy CONNECT object that uses the actual load_data_file and check_likelihood_filter_health functions
    # =============================================================================
    # (These functions are assumed to be exactly as in your CONNECT source code.)
    # You can copy them verbatim from your code if they are not already imported.
    # For this example, we assume they are defined below.


    # --- Provided function: compare_dataframes ---


    def load_data_file(self, file_path, verbose=1):
        """
        Load a data file that has a header line starting with '#' and returns a DataFrame.
        """
        if not os.path.isfile(file_path):
            if verbose >= 1:
                print(f"[load_data_file] File {file_path} does not exist.", flush=True)
            return None

        header_line = None
        with open(file_path, "r") as f:
            for line in f:
                if line.startswith("#"):
                    header_line = line.lstrip("#").strip()
                    break

        if header_line is None:
            raise ValueError(f"No header line starting with '#' found in {file_path}")

        columns = header_line.split()
        if verbose >= 3:
            print(f"[load_data_file] Columns for {file_path}: {columns}", flush=True)

        df = pd.read_csv(
            file_path,
            sep=r"\s+",
            comment="#",
            names=columns,
            index_col=False,
            dtype=np.float32,
        )

        # Optional sanity checks
        if df.empty and verbose >= 2:
            print(
                f"[load_data_file] Warning: Loaded DataFrame from {file_path} is empty.",
                flush=True,
            )

        return df


    def compare_dataframes(
        df1,
        df2,
        df_likelihood=None,
        df_likelihood2=None,
        comparison_type="new",
        verbose=1,
        compare_context=None,
    ):

        import pandas as pd

        """
        
        #######
        This function was initially developed for the 'plot_iterations.py' module used to analyze the iterative sampling process.
        But it can be used here as well to compare the data overlap between the final accepted data by the likelihood-filter and the percentage of new data from the chains.
        This gives information of how large percentage of the data is actually accepted, and helps us track if the likelihood-filter is too strict for the procedure to converge naturally.
        #######

        Compare two DataFrames (df1, df2) to identify samples that are 'new', 'removed', or 'common',
        while preserving one-to-one matching of duplicates. Also re-aligns likelihood data if provided.

        Parameters
        ----------
        df1 : pd.DataFrame
            The first DataFrame (e.g., the "current" iteration's data).
        df2 : pd.DataFrame
            The second DataFrame (e.g., the "previous" iteration's data).
        df_likelihood : pd.DataFrame, optional
            Likelihood rows aligned with df1 (same length, same row order as df1 BEFORE sorting).
        df_likelihood2 : pd.DataFrame, optional
            Likelihood rows aligned with df2 (same length, same row order as df2 BEFORE sorting).
        comparison_type : {'new','removed','common'}, default='new'
            - 'new': return rows in df1 that are not in df2.
            - 'removed': return rows in df2 that are not in df1.
            - 'common': return rows present in both df1 and df2.
        verbose : int, optional
            If >0, prints some debugging info.

        Returns
        -------
        matched_params : pd.DataFrame
            Subset of parameter rows that match the requested relationship,
            extracted from the correct perspective (df1 or df2, or intersection).
        matched_likelihood : pd.DataFrame or None
            Subset of likelihood rows that align with matched_params. If none given,
            returns None.
        """

        # --------------------------------------------------
        # Step 1: Validate input parameters
        # --------------------------------------------------
        valid_types = ["new", "removed", "common"]
        if comparison_type not in valid_types:
            raise ValueError(
                f"comparison_type must be one of {valid_types}, got: {comparison_type}"
            )

        # --------------------------------------------------
        # Step 2: Handle edge cases (if df1 or df2 is empty)
        # --------------------------------------------------
        if df1 is None or df1.empty:
            if verbose > 0:
                print(
                    f'\n[compare_dataframes] [{compare_context["context"]}] df1 ({compare_context["df1"]}) is empty; returning trivial result:\n {compare_context["msg1"]}'
                )
            if comparison_type == "removed" and df2 is not None:
                return df2.copy().reset_index(drop=True), df_likelihood2
            return None, None

        if df2 is None or df2.empty:
            if verbose > 0:
                print(
                    f'\n[compare_dataframes] [{compare_context["context"]}] df2 ({compare_context["df2"]}) is empty; returning trivial result:\n {compare_context["msg2"]}'
                )
            if comparison_type == "new":
                return df1.copy().reset_index(drop=True), df_likelihood
            return None, None

        # --------------------------------------------------
        # Step 2b: Ensure likelihood data and dataframes match in length
        if df_likelihood is not None and len(df_likelihood) != len(df1):
            raise ValueError(
                f'\nLength mismatch: df_likelihood ({len(df_likelihood)}) and df1 ({compare_context["df1"]}) ({len(df1)}) are not equal.'
            )
        if df_likelihood2 is not None and len(df_likelihood2) != len(df2):
            raise ValueError(
                f'\nLength mismatch: df_likelihood2 ({len(df_likelihood2)}) and df2 ({compare_context["df2"]}) ({len(df2)}) are not equal.'
            )

        # --------------------------------------------------
        # Step 3: Preprocess df1 (current iteration's data)
        # --------------------------------------------------
        df1 = df1.copy()
        df1["_temp_idx1"] = df1.index  # Store original row index before sorting

        # Identify the relevant parameter columns (excluding helper columns)
        param_cols = [c for c in df1.columns if c not in ["_temp_idx1", "dup_id"]]

        # Sort df1 so that identical samples appear together
        df1_sorted = df1.sort_values(param_cols, kind="mergesort").reset_index(drop=True)

        # Assign a 'dup_id' to each duplicate row so they can be matched one-to-one
        df1_sorted["dup_id"] = df1_sorted.groupby(param_cols).cumcount()

        # Reorder the likelihood data to match this new sorted order
        df_likelihood_sorted = (
            df_likelihood.iloc[df1_sorted["_temp_idx1"]].reset_index(drop=True)
            if df_likelihood is not None
            else None
        )

        # --------------------------------------------------
        # Step 4: Preprocess df2 (previous iteration's data)
        # --------------------------------------------------
        df2 = df2.copy()
        df2["_temp_idx2"] = df2.index

        df2_sorted = df2.sort_values(param_cols, kind="mergesort").reset_index(drop=True)
        df2_sorted["dup_id"] = df2_sorted.groupby(param_cols).cumcount()

        df_likelihood2_sorted = (
            df_likelihood2.iloc[df2_sorted["_temp_idx2"]].reset_index(drop=True)
            if df_likelihood2 is not None
            else None
        )

        # --------------------------------------------------
        # Step 5: Perform Merge to Find Matches
        # --------------------------------------------------
        """
        We now compare df1_sorted and df2_sorted to determine which samples belong to which category:
        
        - 'new': Samples in df1 but not in df2 (found using a LEFT JOIN)
        - 'removed': Samples in df2 but not in df1 (found using a RIGHT JOIN)
        - 'common': Samples that exist in both df1 and df2 (found using an INNER JOIN)

        The 'merge' function combines both dataframes based on their common parameter columns + 'dup_id'.
        This ensures that duplicate rows match correctly and one-to-one.
        
        The 'how' parameter controls which type of comparison we perform:
        
        - 'left' (for 'new'): Keeps all rows from df1_sorted, adds matches from df2_sorted.
        - 'right' (for 'removed'): Keeps all rows from df2_sorted, adds matches from df1_sorted.
        - 'inner' (for 'common'): Keeps only rows that exist in BOTH df1_sorted and df2_sorted.

        The 'indicator=True' adds a new column `_merge`, which labels each row as:
        - 'left_only'  → Present only in df1 (new sample)
        - 'right_only' → Present only in df2 (removed sample)
        - 'both'       → Present in both (common sample)
        """
        if comparison_type == "new":
            merge_type = "left"
            indicator = True
        elif comparison_type == "removed":
            merge_type = "right"
            indicator = True
        else:  # 'common'
            merge_type = "inner"
            indicator = False

        merged = df1_sorted.merge(
            df2_sorted,
            on=param_cols + ["dup_id"],
            how=merge_type,
            indicator=indicator,
            suffixes=("_df1", "_df2"),
        )

        # --------------------------------------------------
        # Step 6: Extract the Matching Rows from the Merge
        # --------------------------------------------------
        """
        Now that we have merged df1_sorted and df2_sorted, we extract the rows based on `_merge`:

        - For 'new': We filter only rows labeled as 'left_only' (i.e., samples that appear in df1 but not df2).
        - For 'removed': We filter only rows labeled as 'right_only' (samples in df2 but not df1).
        - For 'common': We take all merged rows, since they exist in both dataframes.
        """
        if comparison_type == "new":
            matched_df = merged[merged["_merge"] == "left_only"].drop(columns=["_merge"])
        elif comparison_type == "removed":
            matched_df = merged[merged["_merge"] == "right_only"].drop(columns=["_merge"])
        else:  # 'common'
            matched_df = merged

        # Extract the original rows from df1 or df2
        matched_params = (
            df1.iloc[matched_df["_temp_idx1"]].copy()
            if comparison_type != "removed"
            else df2.iloc[matched_df["_temp_idx2"]].copy()
        )

        # Remove helper columns
        matched_params.drop(
            columns=["dup_id", "_temp_idx1", "_temp_idx2"],
            inplace=True,
            errors="ignore",
        )
        matched_params.reset_index(drop=True, inplace=True)

        # --------------------------------------------------
        # Step 7: Extract Aligned Likelihood Data
        # --------------------------------------------------
        matched_likelihood = None
        if comparison_type in ["new", "common"] and df_likelihood_sorted is not None:
            matched_likelihood = (
                df_likelihood.iloc[matched_df["_temp_idx1"]].copy().reset_index(drop=True)
            )
        elif comparison_type == "removed" and df_likelihood2_sorted is not None:
            matched_likelihood = (
                df_likelihood2.iloc[matched_df["_temp_idx2"]].copy().reset_index(drop=True)
            )

        return matched_params, matched_likelihood

        # --- Provided function: check_likelihood_filter_health ---

    def print_accepted_percentage(
        self,
        i,
        mcmc,
    ):

        # ---------------- 2nd CONVERGENCE CHECK ----------------
        # Import model_params from current iteration. The accepted points after the likelihood filter
        all_accepted_after_lklfilter_df = self.load_data_file(
            os.path.join(
                self.CONNECT_PATH,
                self.data_path,
                f"number_{i}",
                "model_params.txt",
            ),
            verbose=2,
        )
        all_accepted_after_lklfilter_likelihood_df = self.load_data_file(
            os.path.join(
                self.CONNECT_PATH,
                self.data_path,
                f"number_{i}",
                "likelihood_data.txt",
            ),
            verbose=2,
        )
        N_all_accepted_after_lkl_filter = len(all_accepted_after_lklfilter_df)

        # Import data from chains; i.e. data accepted from oversampling filter
        accepted_by_oversampling = mcmc.import_points_from_chains(i)
        N_accepted_by_oversampling = len(accepted_by_oversampling)
        N_in_data_set = (
            mcmc.get_number_of_data_points(i - 1) + N_accepted_by_oversampling
        )

        param_cols = all_accepted_after_lklfilter_df.columns
        accepted_by_oversampling_df = pd.DataFrame(
            accepted_by_oversampling, columns=param_cols
        )

        # Compare the two dataframes to find the overlap
        compare_context = {
            "context": "Finding overlap between points accepted by likelihood filter and new points from chains",
            "df1": f"all_accepted_after_lklfilter",
            "df2": f"accepted_by_oversampling",
            "msg1": "all_accepted_after_lklfilter is empty",
            "msg2": "accepted_by_oversampling is empty",
        }
        # New data that survived the oversampling filter and also survived the likelihood filter. i.e. all new points added to the accepted pool this iteration.
        new_accepted_df, new_accepted_likelihood_df = compare_dataframes(
            df1=all_accepted_after_lklfilter_df,
            df2=accepted_by_oversampling_df,
            df_likelihood=all_accepted_after_lklfilter_likelihood_df,
            comparison_type="common",  # Find common points between the two dataframes
            compare_context=compare_context,
            verbose=1,
        )

        N_new_accepted_after_lkl_filter = len(new_accepted_df)


        return f"    The new points added constitutes {N_new_accepted_after_lkl_filter/N_all_accepted_after_lkl_filter*100:.1f}% of the total accepted pool after applying the likelihood-filter."



    # Now, define a dummy CONNECT-like object that uses the real functions above.
    class DummyCONNECT:
        def __init__(self, CONNECT_PATH, data_path, param):
            self.CONNECT_PATH = CONNECT_PATH
            self.data_path = data_path
            self.param = param
            self.param.auto_update_threshold = True
            # Set this to trigger the threshold update branch.
            self.consecutive_bad_states_count = 4

        # Attach the functions as methods
        load_data_file = load_data_file
        print_accepted_percentage = print_accepted_percentage
        #check_likelihood_filter_health = check_likelihood_filter_health


    # =============================================================================
    # 3. Instantiate the dummy CONNECT object and the real mcmc sampler
    # =============================================================================
    dummy_connect = DummyCONNECT(CONNECT_PATH, DATA_PATH, param)

    # Instantiate the mcmc sampler using your exec method (this uses the actual mcmc sampler)
    exec(f"from source.mcmc_samplers.{param.mcmc_sampler} import {param.mcmc_sampler}")
    _locals = {}
    exec(
        f"mcmc = {param.mcmc_sampler}(param, CONNECT_PATH)",
        globals(),
        _locals,
    )
    mcmc = _locals["mcmc"]


    #Find iterations based on folders numbers_1, numbers_2, etc. in the data folder:
    iterations = [int(f.split("_")[1]) for f in os.listdir(os.path.join(CONNECT_PATH, DATA_PATH)) if f.startswith("number_")]
    # Sort the iterations
    iterations.sort()

    # For example, set N_accepted=15 and N_in_data_set=100 to force the branch where the threshold is updated.
    messages = {}
    for i in iterations:
        message = dummy_connect.print_accepted_percentage(
        i=i, mcmc=mcmc
        )
        messages[i] = message
        
    print(messages)

#Try to call the function
write_acceptance_rate_to_log("dcdm_baseline_filter_thres10000")


FileNotFoundError: [Errno 2] No such file or directory: 'data/dcdm/dcdm_baseline_filter_thres500/number_9/log.param'

In [5]:
import os
CONNECT_PATH = "/home/maanson/Speciale/connectv2"
DATA_PATH = "data/neff/neff_baseline_nofilter"
PARAM_FILE = os.path.join(CONNECT_PATH, DATA_PATH, "log_connect.param")

# Import the Parameters class from CONNECT's source code.
# (Make sure that the module "source.default_module" is on your PYTHONPATH.)
from source.default_module import Parameters

param = Parameters(PARAM_FILE)

from source.join_output import CreateSingleDataFile
CSDF = CreateSingleDataFile(param, CONNECT_PATH)
CSDF.join()

In [7]:
import os
import re
def recover_bad_state_count_from_log(log_path):
    """
    Returns the most recent consecutive_bad_states_count found in output.log,
    or zero if none found.
    """
    if not os.path.isfile(log_path):
        return 0

    # read entire file
    with open(log_path, "r", encoding="utf-8", errors="replace") as f:
        text = f.read()

    # Split into blocks by iteration
    # This will give you a list of iteration blocks, each starting with
    # 'Beginning iteration no. X' line and continuing until the next iteration or end of file.
    blocks = re.split(r"(?=^Beginning iteration no\.\s*\d+)", text, flags=re.MULTILINE)

    # We only care about the last *complete* iteration block, i.e. one that has "New model is" or "Final model is"
    # near the end. So let's find the last block that has that text.
    complete_blocks = []
    for block in blocks:
        # If block has "New model is" or "Final model is", we call it "complete".
        if re.search(r"(New model is|Final model is)", block):
            complete_blocks.append(block)

    if not complete_blocks:
        # No complete blocks found
        return 0

    last_complete_block = complete_blocks[-1]

    # Now search in last_complete_block for the line:
    # "The "bad state" has occurred for the last (\d+) consecutive iterations."
    # We can do a simple pattern:
    pattern = r'The "bad state" has occurred for the last (\d+) consecutive iterations'
    match = re.search(pattern, last_complete_block)
    if match:
        return int(match.group(1))
    else:
        return 0


# Example usage
log_path = os.path.join("/home/maanson/Speciale/connectv2/data/dcdm/dcdm_baseline_filter_thres3000", "output.log")
#log_path = "/home/maanson/Speciale/connectv2/data/dcdm/dcdm_baseline_filter_thres3000/output.log"
bad_state_count = recover_bad_state_count_from_log(log_path)

print(f"Consecutive bad states count: {bad_state_count}")

Consecutive bad states count: 3
